# Demo: Information Retrieval & Retrieval-Augmented Generation

Demo gồm 3 phần chính:
1. **Sparse Retrieval (Tìm kiếm thưa)**: Sử dụng thuật toán BM25.
2. **Dense Retrieval (Tìm kiếm dày)**: Sử dụng Bi-encoder với FAISS.
3. **Retrieval-Augmented Generation (RAG)**: Kết hợp tài liệu tìm được với LLM để sinh câu trả lời.
   
**Dataset sử dụng:** [Wikipedia Movie Plots](https://www.kaggle.com/datasets/jrobischon/wikipedia-movie-plots) trên Kaggle.

In [1]:
# Cài đặt các thư viện cần thiết
!pip install -q transformers sentence-transformers faiss-cpu rank_bm25 pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.1 MB/s eta 0:00:00


## Bước 1: Tải và chuẩn bị dữ liệu
Tải dataset `wiki_movie_plots_deduped.csv`

In [2]:
import pandas as pd

# Đọc dữ liệu (Đảm bảo file csv nằm cùng thư mục hoặc chỉnh lại đường dẫn)
# Tải dataset tại: https://www.kaggle.com/datasets/jrobischon/wikipedia-movie-plots
df = pd.read_csv('/kaggle/input/datasets/jrobischon/wikipedia-movie-plots/wiki_movie_plots_deduped.csv')

# Lấy 5000 phim đầu tiên để demo cho nhẹ
df_subset = df.head(5000).copy()

# Tạo một danh sách các "documents". Mỗi document là cốt truyện của một bộ phim.
documents = df_subset['Plot'].tolist()
titles = df_subset['Title'].tolist()

print(f"Tổng số tài liệu (documents): {len(documents)}")
print(f"Ví dụ tài liệu đầu tiên (Phim {titles[0]}): {documents[0][:200]}...")

Tổng số tài liệu (documents): 5000
Ví dụ tài liệu đầu tiên (Phim Kansas Saloon Smashers): A bartender is working at a saloon, serving drinks to customers. After he fills a stereotypically Irish man's bucket with beer, Carrie Nation and her followers burst inside. They assault the Irish man...


## Bước 2: Sparse Retrieval với BM25

In [3]:
from rank_bm25 import BM25Okapi

# Tokenize corpus (tách từ cơ bản bằng khoảng trắng)
tokenized_corpus = [doc.lower().split(" ") for doc in documents]

# Khởi tạo mô hình BM25
bm25 = BM25Okapi(tokenized_corpus)

# Định nghĩa hàm tìm kiếm
def sparse_search(query, top_k=3):
    tokenized_query = query.lower().split(" ")
    # Lấy điểm số của toàn bộ document
    doc_scores = bm25.get_scores(tokenized_query)
    # Lấy top k documents tốt nhất
    top_docs = bm25.get_top_n(tokenized_query, documents, n=top_k)
    
    print(f"--- KẾT QUẢ SPARSE RETRIEVAL (BM25) CHO QUERY: '{query}' ---")
    for i, doc in enumerate(top_docs):
        # Tìm lại index để in ra tên phim
        idx = documents.index(doc)
        print(f"\nTop {i+1} - Phim: {titles[idx]}")
        print(f"Đoạn trích: {doc[:300]}...")

# Chạy thử nghiệm
query = "a giant monster attacks the city"
sparse_search(query)

--- KẾT QUẢ SPARSE RETRIEVAL (BM25) CHO QUERY: 'a giant monster attacks the city' ---

Top 1 - Phim: The Ghost of Frankenstein
Đoạn trích: The residents of the village of Frankenstein feel they are under a curse and blame all their troubles on Frankenstein's monster. The Mayor allows them to destroy Frankenstein's castle. Ygor finds the monster released from his sulfuric tomb by the explosions. The exposure to the sulfur weakened yet p...

Top 2 - Phim: House of Frankenstein
Đoạn trích: Dr. Gustav Niemann (Boris Karloff) escapes from prison along with his hunchbacked assistant Daniel (J. Carrol Naish), for whom he promises to create a new, beautiful body. The two murder Professor Lampini (George Zucco), a traveling showman, and take over his horror exhibit. To exact revenge on Bürg...

Top 3 - Phim: Bride of Frankenstein
Đoạn trích: On a stormy night, Percy Bysshe Shelley (Douglas Walton) and Lord Byron (Gavin Gordon) praise Mary Shelley (Elsa Lanchester) for her story of Frankenstein 

## Bước 3: Dense Retrieval với Bi-encoder và FAISS 

In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Sử dụng mô hình bi-encoder nhỏ, tối ưu cho tốc độ
retriever_model = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Encode toàn bộ tài liệu thành dense vectors (sẽ mất khoảng 1-2 phút)
print("Đang encode documents sang dense vectors...")
doc_embeddings = retriever_model.encode(documents, show_progress_bar=True)

# 2. Xây dựng index bằng FAISS để tìm kiếm nearest neighbor
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings) # Đưa vector vào FAISS index

def dense_search(query, top_k=3):
    # Encode câu truy vấn
    query_embedding = retriever_model.encode([query])
    # Tìm kiếm trong FAISS
    distances, indices = index.search(query_embedding, top_k)
    
    print(f"--- KẾT QUẢ DENSE RETRIEVAL CHO QUERY: '{query}' ---")
    for i in range(top_k):
        idx = indices[0][i]
        print(f"\nTop {i+1} - Phim: {titles[idx]} (Distance: {distances[0][i]:.4f})")
        print(f"Đoạn trích: {documents[idx][:300]}...")
        
    return indices[0] # Trả về list index để dùng cho RAG phía dưới

# Chạy thử nghiệm với câu truy vấn ngữ nghĩa
query = "a scientist creates a creature from dead body parts" # Ám chỉ Frankenstein
top_doc_indices = dense_search(query)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Đang encode documents sang dense vectors...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

--- KẾT QUẢ DENSE RETRIEVAL CHO QUERY: 'a scientist creates a creature from dead body parts' ---

Top 1 - Phim: Life Returns (Distance: 0.9631)
Đoạn trích: A doctor who is convinced that the dead can be brought back to life gets the chance to prove his theory on a dog that has recently died....

Top 2 - Phim: Frankenstein (Distance: 1.0292)
Đoạn trích: In a European village, a young scientist, named Henry Frankenstein, and his assistant Fritz, a hunchback, piece together a human body, the parts of which have been collected from various sources, including stealing freshly buried bodies in a cemetery, and recently hanged criminals. Frankenstein desi...

Top 3 - Phim: Maniac (Distance: 1.1555)
Đoạn trích: Don Maxwell is a former vaudeville impersonator who is working as the lab assistant to Dr. Meirschultz, a mad scientist attempting to bring the dead back to life. When Don kills Meirschultz, he attempts to hide his crime by "becoming" the doctor, taking over his work and copying his appe

## Bước 4: Retrieval-Augmented Generation (RAG)

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# 1. Xác định thiết bị (Dùng GPU nếu có để mô hình generate nhanh hơn)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Khởi tạo Tokenizer và Model
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device) # Đưa model lên GPU/CPU

def rag_pipeline(question):
    print(f"\nCâu hỏi của người dùng: {question}")
    
    # Bước 1: Retrieval
    query_embedding = retriever_model.encode([question])
    _, indices = index.search(query_embedding, 1)
    
    retrieved_idx = indices[0][0]
    retrieved_doc = documents[retrieved_idx]
    movie_title = titles[retrieved_idx]
    
    # XỬ LÝ LỖI CẮT CHUỖI:
    # Thay vì cắt bằng số ký tự (dễ gây đứt ngang chữ), ta cắt theo số lượng từ (words).
    # Lấy 300 từ đầu tiên (~400 tokens, vừa vặn với T5).
    words = retrieved_doc.split()
    truncated_doc = " ".join(words[:300]) 
    
    # Bước 2: Xây dựng Prompt (Viết rõ ràng để chỉ dẫn FLAN-T5 tốt hơn)
    prompt = f"Answer the question based on the context below.\n\nContext: {truncated_doc}\n\nQuestion: {question}\nAnswer:"

    # Bước 3: Generate
    # Đưa tokenizer vào với truncation=True và max_length=512 để dọn dẹp các token bị lố
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=512
    ).to(device) # Đưa tensors lên cùng thiết bị với Model
    
    # Điều chỉnh Generation Configs:
    # - do_sample=False: Ép mô hình dùng Greedy Decoding (luôn chọn từ có xác suất cao nhất) -> Tốt cho tác vụ Extract Fact
    # - early_stopping=True: Dừng ngay khi sinh xong câu, không lặp ký tự.
    outputs = model.generate(
        **inputs, 
        max_new_tokens=50, 
        do_sample=False,      
        num_beams=2,          # Beam search nhẹ để câu trả lời trau chuốt hơn
        early_stopping=True
    )
    
    # Giải mã output tensor
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print("\n" + "="*60)
    print(f"Tài liệu tìm thấy (Nguồn: phim {movie_title})")
    print("-" * 60)
    print(f"Câu trả lời từ RAG LLM: {answer}")
    print("="*60)

# Thử nghiệm RAG
user_question_1 = "What is the name of the monster created from dead body parts?"
rag_pipeline(user_question_1)

user_question_2 = "Who is the main character in the Wizard of Oz?"
rag_pipeline(user_question_2)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Câu hỏi của người dùng: What is the name of the monster created from dead body parts?

Tài liệu tìm thấy (Nguồn: phim The Son of Frankenstein)
------------------------------------------------------------
Câu trả lời từ RAG LLM: Ygor

Câu hỏi của người dùng: Who is the main character in the Wizard of Oz?

Tài liệu tìm thấy (Nguồn: phim The Wonderful Wizard of Oz)
------------------------------------------------------------
Câu trả lời từ RAG LLM: The Wizard
